# Predicting chicory adulteration from SWIR reflectance

Builds a PLS regression for `chicory_pct` from `training_samples.csv`, with a graph at every stage so you
can see what each step does rather than take it on trust.

## What we already know from the data

Three findings from the diagnostics drive the design:

**1. The two sessions differ by a uniform brightness factor of 1.147.** Day 2 is 14.7 % brighter than day 1.
The ratio has a standard deviation of only 0.020 across all bands, so it is multiplicative — packing
density or sample height, not chemistry. Band depths, which are scale-free, are linear with chicory
content at r = −0.99 and extrapolate to within 7 % of the day-1 plates. **The days are chemically
consistent; only the brightness scale moved.**

This matters because your calibration mixtures are all day 2 while RB100 and all three unknowns are day 1.
Left uncorrected, a day-2 model predicts day-1 pure robusta at **+10.5 % chicory** when the truth is 0.

**2. Working in absorbance fixes it.** `A = log10(1/R)` turns a multiplicative factor into an additive
offset; subtracting each spectrum's own mean removes it. That drops the day bias from +10.5 to −1.6 pp.
Derivatives make it *worse* (SG1 pushes it to +20 pp), so they are not used here.

**3. The signal is real and in the right place.** Within day 2 the bands correlating most strongly with
chicory content are 1704–1764 nm at r = 0.91 — the C-H first overtone. Coffee carries 10–15 % oil, chicory
almost none. That is chemistry, not artefact.

## Design decisions

- **One target.** Within the series robusta + chicory = 100, so `robusta_pct = 100 − chicory_pct` falls
  out of a single model. That identity only holds if the sample really is robusta and chicory — arabica
  breaks it, which is what the applicability domain in section 8 is for.
- **Calibration is 0–50 % chicory, six levels.** CH100 is excluded: there is a 50-point gap below it and
  including it would let one plate dictate the slope. Pooling it gives RMSECV 8.7 pp with CH100 itself
  biased −15 pp.
- **PLS, four latent variables.** 288 collinear predictors and **six distinct concentrations**. Not 2,660
  — six. Flexible models (forests, boosting, neural nets) would fit plate identity and report a score that
  means nothing.
- **1000–2450 nm only.** Below 1000 nm is the noisy detector edge, where the largest apparent RB100/CH100
  difference sits; above 2450 nm the Spectralon certificate has run out.

## 1. Setup and loading

In [ ]:
import json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

from scipy.signal import savgol_filter
from scipy.stats import f as f_dist, chi2
from scipy.optimize import nnls

from sklearn.cross_decomposition import PLSRegression
from sklearn.decomposition import PCA
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import mean_squared_error, r2_score

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 110, "font.size": 9, "figure.facecolor": "white",
                     "axes.grid": True, "grid.alpha": 0.3})

# ------------------------------- EDIT -------------------------------
CSV_DIR = Path(".")            # folder holding training_samples.csv and wavelengths.csv
OUT     = Path("./model_out")
WL_MIN, WL_MAX = 1000.0, 2450.0
CAL_MAX_CHICORY = 50.0         # calibration range
N_PERM  = 300                  # permutation test repeats
# ---------------------------------------------------------------------

OUT.mkdir(exist_ok=True, parents=True)
RNG = np.random.default_rng(0)

df = pd.read_csv(CSV_DIR / "training_samples.csv")
wl_cols_all = [c for c in df.columns if c.replace(".", "", 1).isdigit()]
wl_all = np.array([float(c) for c in wl_cols_all])
keep = (wl_all >= WL_MIN) & (wl_all <= WL_MAX)
WL = wl_all[keep]
wl_cols = [c for c, k in zip(wl_cols_all, keep) if k]

R = np.clip(df[wl_cols].values.astype(float), 1e-3, None)
meta = df[[c for c in df.columns if c not in wl_cols_all]].copy()

print(f"{len(df)} samples | {len(wl_all)} bands -> {len(WL)} kept ({WL[0]:.0f}-{WL[-1]:.0f} nm)")
print(f"dropped {int((~keep).sum())} bands outside the working range")
print()
print(meta.groupby(["role", "label"]).agg(samples=("sample_id", "size"),
                                          plates=("group", "nunique")).to_string())

In [ ]:
# what the trim removes
fig, axs = plt.subplots(1, 2, figsize=(13, 3.8))
Rall = np.clip(df[wl_cols_all].values.astype(float), 1e-3, None)
for lab in ["RB100", "CH100", "AB100"]:
    s = (meta.label == lab).values
    axs[0].plot(wl_all, Rall[s].mean(0), lw=1.3, label=lab)
axs[0].axvspan(wl_all[0], WL_MIN, color="crimson", alpha=.13)
axs[0].axvspan(WL_MAX, wl_all[-1], color="crimson", alpha=.13)
axs[0].set_title("raw reflectance (shaded = discarded)"); axs[0].legend(fontsize=7)

rb = Rall[(meta.label == "RB100").values].mean(0)
ch = Rall[(meta.label == "CH100").values].mean(0)
def snv1(v): return (v - v.mean()) / v.std()
axs[1].plot(wl_all, snv1(rb) - snv1(ch), lw=1.3, color="purple")
axs[1].axvspan(wl_all[0], WL_MIN, color="crimson", alpha=.13)
axs[1].axvspan(WL_MAX, wl_all[-1], color="crimson", alpha=.13)
axs[1].axhline(0, color="k", lw=.6)
axs[1].set_title("RB100 − CH100 after SNV: the biggest gap is on the noisy edge")
for ax in axs: ax.set_xlabel("wavelength (nm)")
plt.tight_layout(); plt.savefig(OUT / "01_band_trim.png", dpi=130, bbox_inches="tight"); plt.show()
print("The spike below 1000 nm is detector noise, not chemistry. Keeping it would let the model")
print("separate the two pure materials using nothing but the low-SNR edge.")

## 2. Preprocessing

Six candidates. The one that matters for your data is **absorbance + offset removal**, for the reason in
the header: your session difference is a multiplicative brightness factor, and `log10(1/R)` turns that into
an additive offset that subtracting the row mean deletes exactly.

SNV divides by the standard deviation as well. That flatters within-day cross-validation — it also
squashes genuine composition-driven contrast — but it does not transfer across sessions, which is what the
next section demonstrates.

In [ ]:
def snv(A):    return (A - A.mean(1, keepdims=True)) / (A.std(1, keepdims=True) + 1e-12)
def offset(A): return A - A.mean(1, keepdims=True)          # additive removal only
def absorb(A): return np.log10(1.0 / A)
def sg(A, d, w=15, p=2): return savgol_filter(A, w, p, deriv=d, axis=1)
def msc(A, ref=None):
    ref = A.mean(0) if ref is None else ref
    M = np.column_stack([ref, np.ones_like(ref)]); out = np.empty_like(A)
    for i, x in enumerate(A):
        (m, c), *_ = np.linalg.lstsq(M, x, rcond=None)
        out[i] = (x - c) / (m if abs(m) > 1e-9 else 1e-9)
    return out

PREP = {
    "raw":                    lambda A: A,
    "SNV":                    lambda A: snv(A),
    "MSC":                    lambda A: msc(A),
    "absorbance":             lambda A: absorb(A),
    "absorbance+offset":      lambda A: offset(absorb(A)),
    "absorbance+SG1":         lambda A: sg(absorb(A), 1),
}

cal = (meta.role == "calibration").values
y_all = meta.chicory_pct.values

fig, axs = plt.subplots(2, 3, figsize=(14, 6))
cmap = plt.get_cmap("viridis")
for ax, (name, f) in zip(axs.ravel(), PREP.items()):
    Z = f(R)
    for lv in sorted(meta.loc[cal, "chicory_pct"].unique()):
        s = cal & (y_all == lv)
        ax.plot(WL, Z[s].mean(0), lw=1.1, color=cmap(lv / 60), label=f"{lv:.0f}%")
    ax.set_title(name, fontsize=9); ax.set_xlabel("nm")
axs[0][0].legend(fontsize=6, ncol=2, title="chicory")
fig.suptitle("each preprocessing applied to the calibration means", fontsize=10)
plt.tight_layout(); plt.savefig(OUT / "02_preprocessing.png", dpi=130, bbox_inches="tight"); plt.show()

## 3. The day effect, seen and measured

Two views of the same problem.

**PCA scores** are unsupervised, so whatever separates the samples there is what genuinely dominates the
variance. Under raw or SNV preprocessing you should see day 1 and day 2 forming two clouds — and that
separation competing with, or beating, the concentration ordering. Under absorbance + offset the clouds
should sit on top of each other while the concentration gradient survives.

**The transfer test** puts a number on it: fit on day 2 alone, then predict day-1 RB100, whose true value
is 0 % chicory. Whatever it returns is the session bias in the units you care about.

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(14, 7))
for col, name in enumerate(["SNV", "absorbance", "absorbance+offset"]):
    Z = PREP[name](R)
    T = PCA(n_components=3).fit_transform(Z)
    for d, mk, cl in [(1, "o", "tab:blue"), (2, "^", "tab:orange")]:
        s = (meta.day == d).values
        axs[0][col].scatter(T[s, 0], T[s, 1], s=4, marker=mk, alpha=.35, c=cl, label=f"day {d}")
    axs[0][col].set_title(f"{name}\ncoloured by DAY", fontsize=9); axs[0][col].legend(fontsize=7)
    sc = axs[1][col].scatter(T[cal, 0], T[cal, 1], s=4, c=y_all[cal], cmap="viridis")
    plt.colorbar(sc, ax=axs[1][col], label="chicory %")
    axs[1][col].set_title("calibration, coloured by concentration", fontsize=9)
    for r_ in (0, 1): axs[r_][col].set_xlabel("PC1"); axs[r_][col].set_ylabel("PC2")
plt.tight_layout(); plt.savefig(OUT / "03_pca_day.png", dpi=130, bbox_inches="tight"); plt.show()
print("Top row: if the two day clouds overlap, the session effect has been removed.")
print("Bottom row: the concentration gradient must survive that removal, or you have thrown away")
print("the signal along with the artefact.")

In [ ]:
# transfer test: fit on day 2, predict day-1 RB100 (true 0%)
d2   = ((meta.day == 2) & meta.chicory_pct.notna()).values
rb0  = (meta.label == "RB100").values
ch100 = (meta.label == "CH100").values
y2 = y_all[d2]

rows = []
for name, f in PREP.items():
    Z = f(R)
    for nc in (3, 4, 5):
        m = PLSRegression(n_components=nc).fit(Z[d2], y2)
        p = np.full(d2.sum(), np.nan)
        for tr, te in LeaveOneGroupOut().split(Z[d2], y2, groups=y2):
            p[te] = PLSRegression(n_components=nc).fit(Z[d2][tr], y2[tr]).predict(Z[d2][te]).ravel()
        rows.append(dict(prep=name, n_comp=nc,
                         rmsecv=float(np.sqrt(mean_squared_error(y2, p))),
                         day_bias=float(m.predict(Z[rb0]).ravel().mean()),
                         ch100=float(m.predict(Z[ch100]).ravel().mean())))
trans = pd.DataFrame(rows)
print(trans.round(2).to_string(index=False))

best_by_bias = trans.loc[trans.day_bias.abs().idxmin()]
fig, axs = plt.subplots(1, 2, figsize=(13, 3.8))
sub = trans[trans.n_comp == 4]
axs[0].barh(sub.prep, sub.day_bias, color=["crimson" if abs(v) > 5 else "seagreen" for v in sub.day_bias])
axs[0].axvline(0, color="k", lw=1); axs[0].set_xlabel("predicted chicory % for RB100 (truth = 0)")
axs[0].set_title("session bias by preprocessing (4 LV)")
for _, r_ in trans.iterrows():
    axs[1].scatter(r_.rmsecv, abs(r_.day_bias), s=34,
                   color="seagreen" if r_.prep == "absorbance+offset" else "0.6")
    axs[1].annotate(f"{r_.prep[:12]}·{r_.n_comp}", (r_.rmsecv, abs(r_.day_bias)), fontsize=5.5,
                    xytext=(3, 3), textcoords="offset points")
axs[1].set_xlabel("RMSECV within day 2 (pp)"); axs[1].set_ylabel("|session bias| (pp)")
axs[1].set_title("the trade-off: best CV is not best transfer")
plt.tight_layout(); plt.savefig(OUT / "04_day_bias.png", dpi=130, bbox_inches="tight"); plt.show()
print(f"\nsmallest session bias: {best_by_bias.prep} with {int(best_by_bias.n_comp)} LV "
      f"-> {best_by_bias.day_bias:+.1f} pp, RMSECV {best_by_bias.rmsecv:.2f}")
print("Because your unknowns are day 1 and your mixtures are day 2, transfer matters more here")
print("than a within-day CV score. Pick for the left-hand axis, not the bottom one.")

## 4. Choosing the configuration

Cross-validation is **leave-one-concentration-out**: every patch of one level is held out, the model is
fitted on the rest, and the held-out level is predicted. That is the only split where the held-out
concentration is genuinely absent from training, so it answers "does this track composition?" rather than
"can it recognise a plate it has already seen?"

Patches from one plate are pseudo-replicates — same powder, same instant. A random split would place
siblings on both sides and report something near perfect.

The two end levels, 0 % and 50 %, are extrapolations in this scheme and will look worse. They are reported
separately rather than averaged in, because hiding them would overstate the interior performance.

In [ ]:
y = y_all[cal]
lvl = y.copy()                      # leave-one-concentration-out groups
plate = meta.loc[cal, "group"].values
capture = meta.loc[cal, "capture"].values

def cv_pred(Z, groups, nc):
    p = np.full(len(y), np.nan)
    for tr, te in LeaveOneGroupOut().split(Z, y, groups=groups):
        p[te] = PLSRegression(n_components=nc).fit(Z[tr], y[tr]).predict(Z[te]).ravel()
    return p

grid = []
for name, f in PREP.items():
    Z = f(R)[cal]
    for nc in range(1, 7):
        p = cv_pred(Z, lvl, nc)
        grid.append(dict(prep=name, n_comp=nc, rmsecv=float(np.sqrt(mean_squared_error(y, p)))))
G = pd.DataFrame(grid)
piv = G.pivot(index="prep", columns="n_comp", values="rmsecv")

fig, ax = plt.subplots(figsize=(7.5, 3.2))
im = ax.imshow(piv.values, cmap="viridis_r", aspect="auto")
ax.set_xticks(range(piv.shape[1]), piv.columns); ax.set_yticks(range(piv.shape[0]), piv.index)
for i in range(piv.shape[0]):
    for j2 in range(piv.shape[1]):
        ax.text(j2, i, f"{piv.values[i, j2]:.1f}", ha="center", va="center", fontsize=6.5,
                color="w" if piv.values[i, j2] > np.nanmedian(piv.values) else "k")
plt.colorbar(im, label="RMSECV (pp)"); ax.set_xlabel("latent variables")
ax.set_title("leave-one-concentration-out RMSECV"); ax.grid(False)
plt.tight_layout(); plt.savefig(OUT / "05_grid.png", dpi=130, bbox_inches="tight"); plt.show()

PREP_NAME, N_COMP = "absorbance+offset", 4        # chosen for transfer, see section 3
print(f"using {PREP_NAME} with {N_COMP} latent variables")
print(f"  its leave-one-concentration-out RMSECV: "
      f"{G[(G.prep==PREP_NAME)&(G.n_comp==N_COMP)].rmsecv.iloc[0]:.2f} pp")
print(f"  best CV in the whole grid:             {G.rmsecv.min():.2f} pp "
      f"({G.loc[G.rmsecv.idxmin(),'prep']}, {int(G.loc[G.rmsecv.idxmin(),'n_comp'])} LV)")
print("\nThe grid winner is not chosen, deliberately -- it fails the transfer test in section 3,")
print("and every sample you actually want to predict comes from the other session.")

## 5. The fitted model

Three things to look at.

**Measured vs predicted** — patches as a cloud, plate means with error bars on top. Judge the plate means;
the cloud shows within-plate scatter, which is surface texture rather than model error.

**Residual diagnostics** — residuals against concentration, stir repeat and patch position. A slope
against patch position would mean an illumination gradient is leaking into the prediction. A step between
stirs would mean the repeats are not exchangeable.

**Regression coefficients and VIP** — the chemical sanity check, and worth more than the R². The raw
correlation with chicory content peaks at 1704–1764 nm, so the model should put weight near there. Weight
sitting on a featureless slope instead would mean it is reading scattering, and any good score is an
accident.

In [ ]:
Zc = PREP[PREP_NAME](R)[cal]
pred_cv = cv_pred(Zc, lvl, N_COMP)
final = PLSRegression(n_components=N_COMP).fit(Zc, y)

rmsecv = float(np.sqrt(mean_squared_error(y, pred_cv)))
interior = ~np.isin(y, [y.min(), y.max()])
fold_tbl = pd.DataFrame([dict(level=lv, n=int((y == lv).sum()),
                              bias=float(np.mean(pred_cv[y == lv] - lv)),
                              rmse=float(np.sqrt(np.mean((pred_cv[y == lv] - lv) ** 2))),
                              kind="extrapolated" if lv in (y.min(), y.max()) else "interpolated")
                        for lv in sorted(set(y))])
print(fold_tbl.round(2).to_string(index=False))
print(f"\noverall RMSECV                {rmsecv:.2f} pp")
print(f"interior levels only         {np.sqrt(np.mean((pred_cv[interior]-y[interior])**2)):.2f} pp")
print(f"R2 (cross-validated)         {r2_score(y, pred_cv):.3f}")

dd = pd.DataFrame(dict(y=y, p=pred_cv, plate=plate, capture=capture))
pm = dd.groupby(["plate", "y"], as_index=False).agg(mean=("p", "mean"), sd=("p", "std"))

fig = plt.figure(figsize=(13.5, 4))
gs = GridSpec(1, 3, figure=fig)
ax = fig.add_subplot(gs[0, 0])
ax.scatter(y + RNG.normal(0, .7, len(y)), pred_cv, s=4, alpha=.16, label="patches")
ax.errorbar(pm["y"], pm["mean"], yerr=pm["sd"], fmt="o", color="crimson", capsize=3,
            ms=5, lw=1, label="plate mean ± sd")
lims = [-12, 62]; ax.plot(lims, lims, "k--", lw=1); ax.set_xlim(lims); ax.set_ylim(lims)
ax.set_xlabel("true chicory %"); ax.set_ylabel("cross-validated prediction %")
ax.set_title(f"RMSECV = {rmsecv:.2f} pp"); ax.legend(fontsize=7)

ax = fig.add_subplot(gs[0, 1])
res = pred_cv - y
for cap in sorted(set(capture)):
    s = capture == cap
    ax.scatter(y[s] + (cap-2)*1.6, res[s], s=5, alpha=.35, label=f"stir {cap}")
ax.axhline(0, color="k", lw=.8); ax.set_xlabel("true chicory %"); ax.set_ylabel("residual (pp)")
ax.set_title("residuals by level and stir"); ax.legend(fontsize=7)

ax = fig.add_subplot(gs[0, 2])
pc = meta.loc[cal, "patch_col"].values
ax.scatter(pc, res, s=4, alpha=.2)
z = np.polyfit(pc, res, 1); xx = np.linspace(pc.min(), pc.max(), 40)
ax.plot(xx, np.poly1d(z)(xx), "r-", lw=1.6)
ax.axhline(0, color="k", lw=.8)
ax.set_xlabel("patch column (across-track)"); ax.set_ylabel("residual (pp)")
ax.set_title(f"residual vs position: slope {z[0]*384:+.1f} pp across the swath")
plt.tight_layout(); plt.savefig(OUT / "06_fit.png", dpi=130, bbox_inches="tight"); plt.show()

In [ ]:
coef = final.coef_.ravel()
if coef.size != WL.size: coef = final.coef_.T.ravel()
W_ = final.x_weights_; T_ = final.x_scores_; Q_ = final.y_loadings_.ravel()
ss = (Q_ ** 2) * (T_ ** 2).sum(0)
vip = np.sqrt(WL.size * ((W_ ** 2 / (W_ ** 2).sum(0)) @ ss) / ss.sum())

BANDS = [(1450, "O-H water"), (1730, "C-H lipid"), (1940, "O-H water"),
         (2100, "C-O carbohydrate"), (2310, "C-H combination")]
fig, axs = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
axs[0].plot(WL, coef, lw=1.2); axs[0].axhline(0, color="k", lw=.6)
axs[0].set_ylabel("PLS coefficient")
axs[1].plot(WL, vip, lw=1.2, color="darkorange"); axs[1].axhline(1, color="r", ls="--", lw=.8)
axs[1].set_ylabel("VIP")
d2m = ((meta.day == 2) & meta.chicory_pct.notna()).values
Zd2 = PREP[PREP_NAME](R)[d2m]
corr = np.array([np.corrcoef(Zd2[:, k], y_all[d2m])[0, 1] for k in range(WL.size)])
axs[2].plot(WL, corr, lw=1.2, color="teal"); axs[2].axhline(0, color="k", lw=.6)
axs[2].set_ylabel("correlation with chicory %"); axs[2].set_xlabel("wavelength (nm)")
for nm, txt in BANDS:
    for a in axs: a.axvline(nm, color="0.7", lw=.8, ls=":")
    axs[0].text(nm, axs[0].get_ylim()[1], txt, fontsize=6, rotation=90, va="top")
plt.tight_layout(); plt.savefig(OUT / "07_coefficients.png", dpi=130, bbox_inches="tight"); plt.show()

top = np.sort(WL[np.argsort(-vip)[:12]])
print("12 most influential wavelengths (nm):", top.astype(int))
print(f"strongest raw correlation: {WL[np.argmax(np.abs(corr))]:.0f} nm, r = {corr[np.argmax(np.abs(corr))]:+.3f}")
print("\nWeight near 1730 nm means lipid -- coffee is 10-15% oil, chicory almost none. Weight near")
print("2100 nm means carbohydrate, and chicory is inulin-rich. Both are the model reading chemistry.")

## 6. Validation

**Leave-one-concentration-out** is the number to report.

**Leave-one-stir-out** measures repeatability: how much a prediction moves when the same powder is
re-stirred and re-sampled. It is not validation — the same concentrations stay in training — but it is the
floor on your precision. No model can be more certain about a sample than re-stirring makes the
measurement.

**The permutation test** is the one people skip. With six levels and eighteen plates, a decent RMSECV can
arise from the model simply separating a handful of objects and happening to order them. Shuffling the
concentrations across whole plates and refitting gives the null distribution for exactly that. If your
real score does not sit clearly in its tail, you do not have evidence of a concentration relationship,
however good the scatter plot looks.

In [ ]:
pred_cap = cv_pred(Zc, capture, N_COMP)
pred_plate = cv_pred(Zc, plate, N_COMP)
rmse_cap = float(np.sqrt(mean_squared_error(y, pred_cap)))
rmse_plate = float(np.sqrt(mean_squared_error(y, pred_plate)))
print(f"leave-one-concentration-out  {rmsecv:6.2f} pp   <- report this")
print(f"leave-one-plate-out          {rmse_plate:6.2f} pp")
print(f"leave-one-stir-out           {rmse_cap:6.2f} pp   (repeatability, not validation)")

plate_y = dd.groupby("plate")["y"].first()
null = []
for _ in range(N_PERM):
    perm = dict(zip(plate_y.index, RNG.permutation(plate_y.values)))
    yp = np.array([perm[g] for g in plate])
    p = np.full(len(yp), np.nan)
    for tr, te in LeaveOneGroupOut().split(Zc, yp, groups=yp):
        p[te] = PLSRegression(n_components=N_COMP).fit(Zc[tr], yp[tr]).predict(Zc[te]).ravel()
    null.append(np.sqrt(mean_squared_error(yp, p)))
null = np.array(null)
pval = (np.sum(null <= rmsecv) + 1) / (len(null) + 1)

fig, axs = plt.subplots(1, 2, figsize=(12, 3.4))
axs[0].bar(["level", "plate", "stir"], [rmsecv, rmse_plate, rmse_cap],
           color=["seagreen", "0.6", "0.6"])
axs[0].set_ylabel("RMSE (pp)"); axs[0].set_title("each split answers a different question")
axs[1].hist(null, bins=30, alpha=.8, color="0.6", label="shuffled labels")
axs[1].axvline(rmsecv, color="crimson", lw=2.2, label=f"real model ({rmsecv:.2f})")
axs[1].set_xlabel("RMSECV (pp)"); axs[1].legend(fontsize=7)
axs[1].set_title(f"permutation test, n={N_PERM}, p = {pval:.3f}")
plt.tight_layout(); plt.savefig(OUT / "08_validation.png", dpi=130, bbox_inches="tight"); plt.show()
print(f"\np = {pval:.4f} -> " + ("the relationship is unlikely to be chance."
      if pval < 0.05 else "NOT significant. Do not report a quantitative model on this data."))

## 7. How much does the calibration choice matter?

A decision has been quietly made: whether RB100 goes into calibration. It is the only 0 % anchor, and it
is the only day-1 plate in the calibration set. Everything else is day 2.

That makes it do two jobs at once — anchor the bottom of the concentration range, *and* be the sole
representative of the other session. Those jobs pull in different directions, so it is worth seeing how
much the answer moves depending on which way you go.

**The spread between the two models is the honest uncertainty from your day confound.** It is not
something the preprocessing removed; it is a consequence of no sample having been measured on both days.

In [ ]:
Zall = PREP[PREP_NAME](R)          # preprocessed spectra for every sample, not just calibration
d2only = ((meta.day == 2) & meta.chicory_pct.notna()).values
MODELS = {}
for name, mask in [("day-2 only (10-50%)", d2only), ("pooled with RB100 (0-50%)", cal)]:
    yy = y_all[mask]
    mm = PLSRegression(n_components=N_COMP).fit(Zall[mask], yy)
    MODELS[name] = mm
    pr = {l: float(mm.predict(Zall[(meta.label == l).values]).ravel().mean())
          for l in ["RB100", "A", "B", "C", "CH100", "AB100"]}
    MODELS[name + "_pred"] = pr
    print(f"{name:26s} " + "  ".join(f"{k}:{v:6.1f}" for k, v in pr.items()))

labs = ["A", "B", "C"]
pa = [MODELS["day-2 only (10-50%)_pred"][l] for l in labs]
pb = [MODELS["pooled with RB100 (0-50%)_pred"][l] for l in labs]
fig, ax = plt.subplots(figsize=(7, 3.6))
x = np.arange(len(labs))
ax.bar(x - .18, pa, .36, label="day-2 only")
ax.bar(x + .18, pb, .36, label="pooled with RB100")
for i in range(len(labs)):
    ax.plot([x[i]]*2, [pa[i], pb[i]], color="crimson", lw=2, zorder=3)
    ax.annotate(f"±{abs(pb[i]-pa[i])/2:.1f}", (x[i], max(pa[i], pb[i])),
                xytext=(0, 6), textcoords="offset points", ha="center", fontsize=8, color="crimson")
ax.axhspan(0, CAL_MAX_CHICORY, color="seagreen", alpha=.10, label="calibration range")
ax.set_xticks(x, labs); ax.set_ylabel("predicted chicory %")
ax.set_title("sensitivity to the calibration choice — the gap is the session uncertainty")
ax.legend(fontsize=7)
plt.tight_layout(); plt.savefig(OUT / "09_sensitivity.png", dpi=130, bbox_inches="tight"); plt.show()

SPREAD = {l: abs(b - a) for l, a, b in zip(labs, pa, pb)}
print("\nhalf-spread per unknown (percentage points):",
      {k: round(v/2, 1) for k, v in SPREAD.items()})
print("Add this to the model error. A swing this large from one extra calibration level means the")
print("session effect is not fully removed, only reduced.")

## 8. Applicability domain — and what it reveals

**Hotelling's T²** is distance from the calibration centre inside the model plane; **Q residual** is
distance off it. High Q means spectral structure the calibration never contained.

Here Q turns out to be doing something more interesting than flagging outliers. Its median is about four
times higher for day 1 than day 2, so it is partly detecting the **session** — the brightness correction
removed the multiplicative part, but a residual spectral difference survives, and every unknown is day 1.

So Q has two contributions mixed together: session and composition. Separating them is what makes it
usable. Ranking the day-1 plates by Q against RB100 — the day-1 plate whose composition *is* in
calibration — removes the session part and leaves the composition part.

Read the ratio, not the absolute value:

- **near 1** — same session character, composition inside the calibrated range;
- **well above 1** — genuinely unlike anything calibrated. CH100 sits here, which is the expected
  behaviour since 100 % chicory is outside the 0–50 % range by design.

In [ ]:
def ad_stats(model, Zfit, Znew, alpha=0.05):
    mu = Zfit.mean(0)
    Tf = (Zfit - mu) @ model.x_rotations_
    Tn = (Znew - mu) @ model.x_rotations_
    Si = np.linalg.pinv(np.atleast_2d(np.cov(Tf, rowvar=False)) + np.eye(Tf.shape[1]) * 1e-12)
    T2 = np.einsum("ij,jk,ik->i", Tn, Si, Tn)
    n, A_ = Tf.shape
    t2lim = A_ * (n - 1) / (n - A_) * f_dist.ppf(1 - alpha, A_, n - A_)
    P = model.x_loadings_
    Qf = np.sum(((Zfit - mu) - Tf @ P.T) ** 2, 1)
    Qn = np.sum(((Znew - mu) - Tn @ P.T) ** 2, 1)
    m_, v_ = Qf.mean(), Qf.var()
    g = v_ / (2 * m_) if m_ > 0 else 1.0
    h = 2 * m_ ** 2 / v_ if v_ > 0 else 1.0
    return T2, t2lim, Qn, g * chi2.ppf(1 - alpha, h)

T2c, t2lim, Qc, qlim = ad_stats(final, Zc, Zc)
Qall = np.full(len(meta), np.nan); T2all = np.full(len(meta), np.nan)
for lab in meta.label.unique():
    s = (meta.label == lab).values
    t2, _, q, _ = ad_stats(final, Zc, Zall[s])
    Qall[s], T2all[s] = q, t2

q_ref = np.median(Qall[(meta.label == "RB100").values])     # the day-1 in-calibration reference

fig, axs = plt.subplots(1, 3, figsize=(15, 4))
for d, c in [(1, "tab:blue"), (2, "tab:orange")]:
    s = (meta.day == d).values & cal
    axs[0].hist(np.log10(Qall[s]), bins=40, alpha=.55, color=c, label=f"day {d}")
axs[0].axvline(np.log10(qlim), color="r", ls="--", lw=1.2, label="95 % limit")
axs[0].set_xlabel("log10 Q residual"); axs[0].set_title("Q separates the two sessions")
axs[0].legend(fontsize=7)

axs[1].scatter(T2c, Qc, s=5, c="0.75", alpha=.4, label="calibration", zorder=0)
for lab in ["A", "B", "C", "CH100", "AB100"]:
    s = (meta.label == lab).values
    axs[1].scatter(T2all[s], Qall[s], s=8, alpha=.55, label=lab)
axs[1].axvline(t2lim, color="r", ls="--", lw=1); axs[1].axhline(qlim, color="r", ls="--", lw=1)
axs[1].set_xscale("log"); axs[1].set_yscale("log")
axs[1].set_xlabel("Hotelling $T^2$"); axs[1].set_ylabel("Q residual")
axs[1].set_title("domain plot"); axs[1].legend(fontsize=6)

d1 = meta[meta.day == 1].assign(Q=Qall[(meta.day == 1).values])
qr = (d1.groupby("label")["Q"].median() / q_ref).sort_values()
axs[2].barh(qr.index, qr.values,
            color=["seagreen" if v < 2 else "crimson" for v in qr.values])
axs[2].axvline(1, color="k", lw=1.2)
axs[2].set_xlabel("median Q ÷ RB100's Q"); axs[2].set_title("day-1 plates, session effect removed")
plt.tight_layout(); plt.savefig(OUT / "10_domain.png", dpi=130, bbox_inches="tight"); plt.show()

print("Q relative to RB100 (day-1 reference, composition inside calibration):")
for lab, v in qr.items():
    tag = "inside calibrated space" if v < 2 else "OUTSIDE -- unlike anything calibrated"
    print(f"   {lab:8s} {v:5.2f}x   {tag}")

## 9. Predicting A, B and C

Each estimate combines three sources of uncertainty, none of which can be dropped:

- **model error** — the leave-one-concentration-out RMSECV;
- **reproducibility** — spread across the three stir repeats;
- **session uncertainty** — the half-spread between the two calibration choices from section 7.

Not the standard error of the patch mean. With hundreds of pseudo-replicate patches that shrinks to a
fraction of a point and would imply a precision you do not have.

`robusta_pct = 100 − chicory_pct` is reported only for samples inside the domain, because the identity
assumes the sample is robusta and chicory and nothing else.

In [ ]:
def summarise(lab):
    s = (meta.label == lab).values
    p = final.predict(Zall[s]).ravel()
    per_cap = pd.Series(p).groupby(meta.loc[s, "capture"].values).mean()
    qrel = float(np.median(Qall[s]) / q_ref)
    u_stir = float(per_cap.std(ddof=1)) if len(per_cap) > 1 else 0.0
    u_sess = SPREAD.get(lab, 0.0) / 2
    u = float(np.sqrt(rmsecv ** 2 + u_stir ** 2 + u_sess ** 2))
    in_dom = qrel < 2.0
    in_rng = -5 <= p.mean() <= CAL_MAX_CHICORY + 5
    if not in_dom and not in_rng:
        verdict = "outside calibrated space AND above the 0-50% range -- treat as 'high chicory', not a number"
    elif not in_dom:
        verdict = "spectrally unlike the calibration set -- do not quote a number"
    elif not in_rng:
        verdict = f"extrapolation beyond {CAL_MAX_CHICORY:.0f}%"
    else:
        verdict = "inside the calibration range"
    return dict(sample=lab, n_patches=int(s.sum()), chicory_pct=round(float(p.mean()), 1),
                uncertainty=round(u, 1), stir_sd=round(u_stir, 2), session_u=round(u_sess, 1),
                Q_rel=round(qrel, 2),
                robusta_pct=round(100 - float(p.mean()), 1) if in_dom else np.nan,
                verdict=verdict)

res = pd.DataFrame([summarise(l) for l in ["A", "B", "C", "RB100", "CH100", "AB100"]])
print(res.to_string(index=False))

fig, axs = plt.subplots(1, 2, figsize=(13, 4))
order = ["A", "B", "C"]
data = [final.predict(Zall[(meta.label == l).values]).ravel() for l in order]
axs[0].violinplot(data, showmeans=True)
axs[0].axhspan(0, CAL_MAX_CHICORY, color="seagreen", alpha=.10, label="calibration range")
axs[0].set_xticks([1, 2, 3], order); axs[0].set_ylabel("predicted chicory %")
axs[0].set_title("spread of patch predictions"); axs[0].legend(fontsize=7)
sub = res[res["sample"].isin(order)]
axs[1].errorbar(range(len(sub)), sub.chicory_pct, yerr=sub.uncertainty, fmt="o", capsize=6, ms=9)
for i, r_ in enumerate(sub.itertuples()):
    axs[1].annotate(f"{r_.chicory_pct:.1f} ± {r_.uncertainty:.1f}", (i, r_.chicory_pct),
                    xytext=(12, 0), textcoords="offset points", fontsize=9)
axs[1].axhspan(0, CAL_MAX_CHICORY, color="seagreen", alpha=.10)
axs[1].set_xticks(range(len(sub)), sub["sample"]); axs[1].set_ylabel("chicory %")
axs[1].set_ylim(0, 100); axs[1].set_title("final estimates with full uncertainty")
plt.tight_layout(); plt.savefig(OUT / "11_predictions.png", dpi=130, bbox_inches="tight"); plt.show()

## 9. Independent cross-check by unmixing

PLS could in principle fit six plates by memorising six objects. This route cannot: it never sees a
concentration label while learning what coffee and chicory look like. It uses only the two pure
endmembers, RB100 and CH100.

Each spectrum is fitted as a non-negative combination of the two pures in **Kubelka–Munk** space,
`f(R) = (1−R)²/2R`, which is roughly additive in concentration for a thick powder where raw reflectance is
not. The resulting abundance is not mass fraction — surface presentation depends on particle size and
packing — so it is calibrated against the known mixtures with a straight line.

If both routes land in the same place on A, B and C, the agreement means something, because they could
have failed independently.

In [ ]:
def km(A): return (1 - np.clip(A, 1e-3, .999)) ** 2 / (2 * np.clip(A, 1e-3, .999))
E = np.column_stack([km(R[(meta.label == "RB100").values]).mean(0),
                     km(R[(meta.label == "CH100").values]).mean(0)])
K = km(R)
frac = np.array([nnls(E, k)[0] for k in K])
frac = 100 * frac[:, 1] / (frac.sum(1) + 1e-12)

tb = meta.assign(ab=frac).groupby(["group", "label", "chicory_pct", "day"], dropna=False,
                                 as_index=False)["ab"].mean()
fit = tb[tb.label.str.startswith("RB[")]
cf = np.polyfit(fit.ab, fit.chicory_pct, 1)
tb["pred_B"] = np.poly1d(cf)(tb.ab)

fig, axs = plt.subplots(1, 2, figsize=(12.5, 4))
axs[0].scatter(fit.ab, fit.chicory_pct, s=45, label="day-2 mixtures")
for lab in ["A", "B", "C", "RB100", "CH100"]:
    s = tb[tb.label == lab]
    if len(s): axs[0].scatter(s.ab, np.poly1d(cf)(s.ab), marker="x", s=70, label=lab)
xx = np.linspace(0, 100, 20)
axs[0].plot(xx, np.poly1d(cf)(xx), "k--", lw=1, label=f"y={cf[0]:.2f}x{cf[1]:+.1f}")
axs[0].plot(xx, xx, color="0.85", lw=1, zorder=0)
axs[0].set_xlabel("chicory abundance from unmixing (%)"); axs[0].set_ylabel("chicory mass %")
axs[0].legend(fontsize=6); axs[0].set_title("Track B calibration")

comp = []
for lab in ["A", "B", "C"]:
    a_ = float(res.loc[res["sample"] == lab, "chicory_pct"].iloc[0])
    b_ = float(tb.loc[tb.label == lab, "pred_B"].mean())
    comp.append((lab, a_, b_))
x = np.arange(len(comp))
axs[1].bar(x - .18, [c[1] for c in comp], .36, label="PLS")
axs[1].bar(x + .18, [c[2] for c in comp], .36, label="unmixing")
axs[1].set_xticks(x, [c[0] for c in comp]); axs[1].set_ylabel("chicory %")
axs[1].legend(fontsize=7); axs[1].set_title("do the two methods agree?")
plt.tight_layout(); plt.savefig(OUT / "11_crosscheck.png", dpi=130, bbox_inches="tight"); plt.show()
for lab, a_, b_ in comp:
    print(f"  {lab}: PLS {a_:5.1f}%   unmixing {b_:5.1f}%   difference {abs(a_-b_):4.1f} pp")
print("\nA gap much larger than the RMSECV means one route is being driven by something other than")
print("composition. Agreement is evidence; disagreement is information.")

In [ ]:
summary = dict(preprocessing=PREP_NAME, n_latent=int(N_COMP),
               wl_range=[float(WL[0]), float(WL[-1])], n_bands=int(WL.size),
               calibration_levels=sorted(map(float, set(y))),
               n_calibration_patches=int(cal.sum()), n_calibration_plates=int(len(set(plate))),
               rmsecv_level=round(rmsecv, 3), rmsecv_plate=round(rmse_plate, 3),
               rmsecv_stir=round(rmse_cap, 3), r2_cv=round(float(r2_score(y, pred_cv)), 4),
               permutation_p=round(float(pval), 4),
               validity_range_pct=[0, CAL_MAX_CHICORY],
               session_spread={k: round(v, 2) for k, v in SPREAD.items()},
               predictions={r["sample"]: dict(chicory_pct=float(r["chicory_pct"]),
                                              robusta_pct=None if pd.isna(r["robusta_pct"])
                                              else float(r["robusta_pct"]),
                                              uncertainty=float(r["uncertainty"]),
                                              Q_rel=float(r["Q_rel"]), verdict=r["verdict"])
                            for _, r in res.iterrows()})
(OUT / "model_summary.json").write_text(json.dumps(summary, indent=2))
res.round(3).to_csv(OUT / "predictions.csv", index=False)
print(json.dumps(summary, indent=2))

## 10. What to report, and what not to claim

**Report:** the leave-one-concentration-out RMSECV with interior and end folds shown separately; the
permutation p-value; the preprocessing and why it was chosen (transfer, not CV score); the measured
session bias; the validity range of 0–50 % chicory; whether each unknown fell inside the applicability
domain; and agreement between PLS and unmixing.

**State plainly:** one preparation per concentration, three stir repeats, six distinct levels. Patches are
pseudo-replicates, so your effective sample size is eighteen calibration plates, not 2,660 rows. Say it
before a reviewer does.

**Do not claim:** an R² from a randomly split patch set; accuracy above 50 % chicory, where you have no
calibration data at all; transferability to arabica — AB100 was measured but never calibrated, and if it
tripped the Q limit that is a finding worth reporting rather than hiding; or a detection limit, unless you
compute one properly and say how.

**The one thing this analysis cannot fix.** No sample was measured on both days. The session correction is
*inferred* from the multiplicative structure of the gap, not *measured* against a transfer standard. It
works — RB100 comes back near 0 % — but it rests on the assumption that the offset is purely
multiplicative. A single plate scanned on both days would turn that assumption into a measurement, and it
is the cheapest improvement available if you ever scan again.